# Food Calorie Regression — End-to-End Pipeline

Single-image (RGB) → total kcal regression, scored by **MAE**. This notebook runs
top-to-bottom on **Google Colab** (target GPU: A100 40GB, degrades gracefully to
L4/T4) and reads/writes all persistent state under a single `PROJECT_DIR` on
Google Drive.

**Design is driven entirely by `analysis.md`** (ground-truth EDA on the actual
3,098 train / 547 test images) and `overview.md` (official rules/format). Every
key statistic used below (per-source medians, skew, duplicate rate, baseline MAE)
is **recomputed from the data on disk and asserted against analysis.md** in the
verification cell — if the data on Drive doesn't match, the notebook fails loudly
instead of silently producing wrong numbers.

**Ground truth recap (see analysis.md for full detail):**
- 3,098 train / 547 test images. Metric = MAE. Test = 30% public / 70% private → trust CV, not LB.
- Two sources, separable by file extension: **A = `.png`**, 640×480 top-down lab rig, 76% of data, median 172 kcal (tight, low). **B = `.jpg`**, 12.2MP oblique smartphone, 24% of data, median 603 kcal (broad, high). **B dominates MAE.**
- Target is right-skewed (skew 3.90); `log1p` symmetrizes it (skew 0.66).
- Baseline to beat: **per-source median ≈ 214.6 MAE** — a model that doesn't clearly beat this per source adds nothing over knowing the file extension.
- 95.7% of labels repeat → near-duplicate dishes → **group-aware CV is mandatory**, plain random K-fold will be optimistic.

**How to use this notebook:**
1. Run Cell 1, set `PROJECT_DIR` to your Drive folder containing `data/` (train/images, test/images, train_labels.csv, test_ids.csv, sample_submission.csv).
2. Leave `SMOKE_TEST = True` in the config cell for the first run — verifies the entire path (cache → train → OOF → inference → submission) in a few minutes on a tiny subset.
3. Set `SMOKE_TEST = False` and re-run top to bottom for the full pipeline. Training is resumable: if Colab disconnects, just re-run — it picks up from the last saved epoch/fold.
4. Final artifact: `PROJECT_DIR/submission.csv`.

## 0. Environment setup

Installs the few packages not preinstalled on Colab: `timm` (backbone zoo),
`imagehash` (perceptual hashing for near-duplicate grouping), and pins
`scikit-learn>=1.1` (needed for `StratifiedGroupKFold`). OpenCV, PIL, torch,
pandas, numpy are already present on Colab.

**Runs on: CPU (Colab, no GPU needed for this cell).**

In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    pip_install(["timm==1.0.*", "imagehash==4.*", "scikit-learn>=1.2"])
else:
    print("Local run — skipping pip install (use your local env).")
print("Environment ready. IN_COLAB =", IN_COLAB)

Environment ready. IN_COLAB = True


## 1. Mount Google Drive & set `PROJECT_DIR`

**Everything persistent lives under `PROJECT_DIR`** on Drive: the competition
data, the resized-image cache, `folds.csv`, downloaded backbone weights,
per-fold checkpoints, OOF predictions, logs, and `submission.csv`. This makes
the whole pipeline resumable across Colab disconnects/reconnects.

Expected layout (edit `PROJECT_DIR` below to point at yours; the competition
data must already exist under `PROJECT_DIR/data/`):

```
PROJECT_DIR/
  data/
    train/images/*.png|*.jpg
    test/images/*.png|*.jpg
    train_labels.csv
    test_ids.csv
    sample_submission.csv
  cache/            (created by this notebook — letterbox image cache)
  weights/          (created by this notebook — offline backbone weights)
  checkpoints/      (created by this notebook — per-fold resumable checkpoints)
  oof/              (created by this notebook — out-of-fold predictions)
  logs/             (created by this notebook)
  folds.csv         (created by this notebook — fixed CV split, saved once)
  submission.csv    (final output)
```

**Runs on: CPU (Colab).**

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/calorie_comp"
    LOCAL_CACHE_DIR = "/content/local_cache"  # fast local copy of the image cache
else:
    PROJECT_DIR = os.path.abspath("local_run")
    LOCAL_CACHE_DIR = f"{PROJECT_DIR}/local_cache"



DATA_DIR = f"{PROJECT_DIR}/data"
CACHE_DIR = f"{PROJECT_DIR}/cache"
WEIGHTS_DIR = f"{PROJECT_DIR}/weights"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
OOF_DIR = f"{PROJECT_DIR}/oof"
LOG_DIR = f"{PROJECT_DIR}/logs"
FOLDS_CSV = f"{PROJECT_DIR}/folds.csv"
SUBMISSION_PATH = f"{PROJECT_DIR}/submission.csv"

print(os.path.exists(f"{PROJECT_DIR}/folds.csv"))
print(os.path.exists(f"{PROJECT_DIR}/weights/convnext_tiny_in1k.pth"))

for d in [CACHE_DIR, WEIGHTS_DIR, CKPT_DIR, OOF_DIR, LOG_DIR, LOCAL_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

required_paths = {
    "train images dir": f"{DATA_DIR}/train/images",
    "test images dir": f"{DATA_DIR}/test/images",
    "train_labels.csv": f"{DATA_DIR}/train_labels.csv",
    "test_ids.csv": f"{DATA_DIR}/test_ids.csv",
    "sample_submission.csv": f"{DATA_DIR}/sample_submission.csv",
}
missing = {name: p for name, p in required_paths.items() if not os.path.exists(p)}
if missing:
    raise FileNotFoundError(
        "PROJECT_DIR/data is missing required paths — fix PROJECT_DIR above.\n"
        + "\n".join(f"  {name}: {p}" for name, p in missing.items())
    )
print("PROJECT_DIR OK:", PROJECT_DIR)
for name, p in required_paths.items():
    print(f"  found {name}: {p}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True
PROJECT_DIR OK: /content/drive/MyDrive/calorie_comp
  found train images dir: /content/drive/MyDrive/calorie_comp/data/train/images
  found test images dir: /content/drive/MyDrive/calorie_comp/data/test/images
  found train_labels.csv: /content/drive/MyDrive/calorie_comp/data/train_labels.csv
  found test_ids.csv: /content/drive/MyDrive/calorie_comp/data/test_ids.csv
  found sample_submission.csv: /content/drive/MyDrive/calorie_comp/data/sample_submission.csv


## 2. Imports & GPU-adaptive settings

Detects the GPU and picks precision + micro-batch size accordingly, while
keeping the **effective batch size fixed** (via gradient accumulation) so the
optimization trajectory doesn't change across GPU tiers:

- **A100 40GB**: `bf16` autocast, micro-batch 64.
- **L4/T4 16GB (Kaggle-like / Colab free)**: `fp16` autocast + `GradScaler`, micro-batch 16.
- **CPU** (no GPU — e.g. this local dry-run): micro-batch 4, fp32, only usable for `SMOKE_TEST` logic checks, not real training.

**Runs on: CPU or GPU (adapts automatically).**

In [ ]:
import os, sys, json, math, time, random, glob, hashlib, shutil, warnings
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=UserWarning)

def detect_device():
    if not torch.cuda.is_available():
        return {"device": torch.device("cpu"), "gpu_name": None, "vram_gb": 0.0,
                "micro_batch": 4, "amp_dtype": torch.float32, "use_grad_scaler": False}
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 35:  # A100 40GB class
        micro_batch, amp_dtype, use_scaler = 64, torch.bfloat16, False
    elif vram_gb >= 20:  # A100 80GB / L40S etc, be generous but still bf16
        micro_batch, amp_dtype, use_scaler = 64, torch.bfloat16, False
    elif vram_gb >= 10:  # T4 / L4 16GB class
        micro_batch, amp_dtype, use_scaler = 16, torch.float16, True
    else:  # small GPU fallback
        micro_batch, amp_dtype, use_scaler = 8, torch.float16, True
    return {"device": torch.device("cuda"), "gpu_name": name, "vram_gb": round(vram_gb, 1),
            "micro_batch": micro_batch, "amp_dtype": amp_dtype, "use_grad_scaler": use_scaler}

GPU_INFO = detect_device()
print("GPU info:", {k: v for k, v in GPU_INFO.items() if k != "device"}, "| device:", GPU_INFO["device"])

GPU info: {'gpu_name': 'NVIDIA A100-SXM4-40GB', 'vram_gb': 39.5, 'micro_batch': 64, 'amp_dtype': torch.bfloat16, 'use_grad_scaler': False} | device: cuda


## 3. Config

Single source of truth for every knob. `IMG_SIZE` and `SMOKE_TEST` are the two
levers you'll actually touch: `IMG_SIZE=384` is the default (matches
`analysis.md`'s recommendation to not downscale Source B too hard); bump to
`448` as an optional lever if CV plateaus and VRAM allows. `SMOKE_TEST=True`
restricts to a tiny random subset, 1 fold, 2 epochs — flip to `False` for the
real run.

`EFFECTIVE_BATCH` is the batch size the *optimizer* sees; the actual per-step
micro-batch and the number of gradient-accumulation steps are derived from the
detected GPU in the previous cell, so this number is stable across GPU tiers.

**Runs on: CPU or GPU.**

In [ ]:
@dataclass
class Config:
    # paths (from cell 1)
    project_dir: str = PROJECT_DIR
    data_dir: str = DATA_DIR
    cache_dir: str = CACHE_DIR
    weights_dir: str = WEIGHTS_DIR
    ckpt_dir: str = CKPT_DIR
    oof_dir: str = OOF_DIR
    log_dir: str = LOG_DIR
    folds_csv: str = FOLDS_CSV
    submission_path: str = SUBMISSION_PATH
    local_cache_dir: str = LOCAL_CACHE_DIR

    # smoke test
    SMOKE_TEST: bool = False
    smoke_per_source: int = 40      # ~40 images per source -> tiny but exercises both sources
    smoke_epochs: int = 2
    smoke_folds: int = 1

    # data / cv
    n_folds: int = 5
    seed: int = 42
    phash_hamming_thresh: int = 4   # out of 64 bits -> near-duplicate cluster threshold
    n_calorie_bins: int = 5         # per-source quantile bins for stratification

    # image
    img_size: int = 384             # config knob; 448 as an optional lever
    pad_value: int = 0              # letterbox pad color (black)

    # model
    backbone_name: str = "convnext_tiny"
    source_emb_dim: int = 16
    head_hidden: int = 256
    head_dropout: float = 0.2

    # optimization
    effective_batch: int = 32
    epochs: int = 100
    warmup_epochs: int = 1
    head_lr: float = 1e-3
    backbone_lr: float = 1e-4
    weight_decay: float = 0.05
    grad_clip_norm: float = 1.0
    loss_type: str = "l1"           # "l1" (default) or "huber" (stability fallback)
    huber_delta: float = 100.0      # kcal
    num_workers: int = 2 if IN_COLAB else 0

    # inference
    tta_dihedral_A: bool = True     # full 8-way dihedral TTA for source A (top-down)
    tta_hflip_B: bool = True        # hflip-only TTA for source B (oblique)

CFG = Config()

if CFG.SMOKE_TEST:
    CFG.epochs = CFG.smoke_epochs
    CFG.n_folds_to_run = CFG.smoke_folds
    CFG.warmup_epochs = 0
    if not IN_COLAB:
        # Local CPU dry-run: exercise the full pipeline in a few minutes.
        CFG.smoke_per_source = min(CFG.smoke_per_source, 8)
        CFG.smoke_epochs = 1
        CFG.epochs = 1
        CFG.tta_dihedral_A = False
        CFG.tta_hflip_B = False
        print("LOCAL CPU dry-run overrides: smoke_per_source=8, epochs=1, TTA off")
else:
    CFG.n_folds_to_run = CFG.n_folds

print(json.dumps({k: (str(v)) for k, v in asdict(CFG).items()}, indent=2))

{
  "project_dir": "/content/drive/MyDrive/calorie_comp",
  "data_dir": "/content/drive/MyDrive/calorie_comp/data",
  "cache_dir": "/content/drive/MyDrive/calorie_comp/cache",
  "weights_dir": "/content/drive/MyDrive/calorie_comp/weights",
  "ckpt_dir": "/content/drive/MyDrive/calorie_comp/checkpoints",
  "oof_dir": "/content/drive/MyDrive/calorie_comp/oof",
  "log_dir": "/content/drive/MyDrive/calorie_comp/logs",
  "folds_csv": "/content/drive/MyDrive/calorie_comp/folds.csv",
  "submission_path": "/content/drive/MyDrive/calorie_comp/submission.csv",
  "local_cache_dir": "/content/local_cache",
  "SMOKE_TEST": "False",
  "smoke_per_source": "40",
  "smoke_epochs": "2",
  "smoke_folds": "1",
  "n_folds": "5",
  "seed": "42",
  "phash_hamming_thresh": "4",
  "n_calorie_bins": "5",
  "img_size": "384",
  "pad_value": "0",
  "backbone_name": "convnext_tiny",
  "source_emb_dim": "16",
  "head_hidden": "256",
  "head_dropout": "0.2",
  "effective_batch": "32",
  "epochs": "100",
  "warmup_ep

## 4. Reproducibility

Seeds Python/`PYTHONHASHSEED`/NumPy/Torch/CUDA, forces deterministic cuDNN,
and enables `torch.use_deterministic_algorithms(warn_only=True)` (warn-only
because a couple of `timm`/cuDNN ops used by ConvNeXt don't have deterministic
kernels yet — we still get determinism everywhere it's available and a loud
warning instead of a silent divergence where it isn't). DataLoaders get a
seeded generator + `worker_init_fn` so batch order and any worker-side
randomness are fixed too. Library versions are logged so a re-run environment
mismatch is visible immediately.

**Runs on: CPU or GPU.**

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(CFG.seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(int(seed))  # int() guards against numpy.int64 (e.g. seed + a fold id from a pandas column)
    return g

import sklearn, timm as _timm
print("Library versions:")
for name, mod in [("python", sys), ("numpy", np), ("pandas", pd), ("torch", torch),
                   ("timm", _timm), ("sklearn", sklearn), ("cv2", cv2)]:
    v = getattr(mod, "__version__", sys.version.split()[0] if mod is sys else "?")
    print(f"  {name}: {v}")
print("cuda available:", torch.cuda.is_available(), "| cuda version:", torch.version.cuda)

Library versions:
  python: 3.12.13
  numpy: 2.0.2
  pandas: 2.2.2
  torch: 2.11.0+cu128
  timm: 1.0.27
  sklearn: 1.6.1
  cv2: 4.13.0
cuda available: True | cuda version: 12.8


## 5. Load competition data & derive source

Source is 100% determined by file extension (`.png` = A, top-down lab rig;
`.jpg` = B, oblique smartphone) — confirmed in `analysis.md` by resolution and
visual inspection. We derive it once here and use it everywhere downstream
(stratification, augmentation branch, source embedding, calibration, clipping).

**Runs on: CPU.**

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train_labels.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test_ids.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

def derive_source(filename: str) -> str:
    ext = filename.rsplit(".", 1)[-1].lower()
    if ext == "png":
        return "A"
    elif ext in ("jpg", "jpeg"):
        return "B"
    raise ValueError(f"Unrecognized extension in filename: {filename}")

train_df["source"] = train_df["filename"].apply(derive_source)
test_df["source"] = test_df["filename"].apply(derive_source)

train_df["path"] = train_df["filename"].apply(lambda fn: f"{DATA_DIR}/train/images/{fn}")
test_df["path"] = test_df["filename"].apply(lambda fn: f"{DATA_DIR}/test/images/{fn}")

missing_train = train_df.loc[~train_df["path"].apply(os.path.exists)]
missing_test = test_df.loc[~test_df["path"].apply(os.path.exists)]
assert len(missing_train) == 0, f"{len(missing_train)} train images listed in train_labels.csv are missing on disk"
assert len(missing_test) == 0, f"{len(missing_test)} test images listed in test_ids.csv are missing on disk"

print(train_df.shape, test_df.shape, sample_sub.shape)
train_df.head()

(3098, 5) (547, 4) (547, 2)


,image_id,filename,calories,source,path
0,train_0000,train_0000.jpg,659.27,B,/content/drive/MyDrive/calorie_comp/data/train...
1,train_0001,train_0001.jpg,635.94,B,/content/drive/MyDrive/calorie_comp/data/train...
2,train_0002,train_0002.png,242.00,A,/content/drive/MyDrive/calorie_comp/data/train...
3,train_0003,train_0003.jpg,591.63,B,/content/drive/MyDrive/calorie_comp/data/train...
4,train_0004,train_0004.png,180.00,A,/content/drive/MyDrive/calorie_comp/data/train...


## 6. Verify recomputed stats against `analysis.md` (ground truth)

These are the exact numbers from `analysis.md` §2–§6. Every one is
**recomputed here from the CSVs/files actually on Drive** and asserted to
match within a small tolerance. If any assertion fails, the data on Drive
differs from what the pipeline was designed for and the notebook **stops
immediately** rather than silently training on the wrong assumptions.

**Runs on: CPU.**

In [ ]:
from scipy.stats import skew as _skew

GROUND_TRUTH = {
    "n_train": 3098, "n_test": 547,
    "n_train_A": 2355, "n_train_B": 743,
    "n_test_A": 417, "n_test_B": 130,
    "median_A": 172.0, "median_B": 603.0,
    "mean_A": 212.5, "mean_B": 914.3,
    "target_min": 50.0, "target_max": 3724.15,
    "global_mean": 380.8, "global_median": 220.0,
    "skew_raw": 3.90, "skew_log1p": 0.66,
    "per_source_median_mae": 214.6,
    "dup_share": 0.957,
    "n_unique_A": 500, "n_unique_B": 207,
}

def assert_close(name, got, expected, tol):
    ok = abs(got - expected) <= tol
    status = "OK" if ok else "MISMATCH"
    print(f"  [{status}] {name}: got={got:.4f} expected={expected} (tol={tol})")
    assert ok, f"GROUND TRUTH MISMATCH on '{name}': got {got}, expected {expected} +/- {tol}. Data on disk may differ from analysis.md."

print("Verifying dataset against analysis.md ground truth...")
assert_close("n_train", len(train_df), GROUND_TRUTH["n_train"], 0)
assert_close("n_test", len(test_df), GROUND_TRUTH["n_test"], 0)

vc_train = train_df["source"].value_counts()
vc_test = test_df["source"].value_counts()
assert_close("n_train_A", vc_train.get("A", 0), GROUND_TRUTH["n_train_A"], 0)
assert_close("n_train_B", vc_train.get("B", 0), GROUND_TRUTH["n_train_B"], 0)
assert_close("n_test_A", vc_test.get("A", 0), GROUND_TRUTH["n_test_A"], 0)
assert_close("n_test_B", vc_test.get("B", 0), GROUND_TRUTH["n_test_B"], 0)

cal = train_df["calories"]
assert_close("target_min", cal.min(), GROUND_TRUTH["target_min"], 0.5)
assert_close("target_max", cal.max(), GROUND_TRUTH["target_max"], 0.5)
assert_close("global_mean", cal.mean(), GROUND_TRUTH["global_mean"], 1.0)
assert_close("global_median", cal.median(), GROUND_TRUTH["global_median"], 0.5)
assert_close("skew_raw", _skew(cal), GROUND_TRUTH["skew_raw"], 0.1)
assert_close("skew_log1p", _skew(np.log1p(cal)), GROUND_TRUTH["skew_log1p"], 0.1)

medians = train_df.groupby("source")["calories"].median()
means = train_df.groupby("source")["calories"].mean()
assert_close("median_A", medians["A"], GROUND_TRUTH["median_A"], 0.5)
assert_close("median_B", medians["B"], GROUND_TRUTH["median_B"], 0.5)
assert_close("mean_A", means["A"], GROUND_TRUTH["mean_A"], 1.0)
assert_close("mean_B", means["B"], GROUND_TRUTH["mean_B"], 1.0)

per_source_median_pred = train_df["source"].map(medians)
per_source_median_mae = (train_df["calories"] - per_source_median_pred).abs().mean()
assert_close("per_source_median_mae", per_source_median_mae, GROUND_TRUTH["per_source_median_mae"], 0.5)

vc_cal = train_df["calories"].value_counts()
dup_share = train_df["calories"].map(vc_cal).gt(1).mean()
assert_close("dup_share", dup_share, GROUND_TRUTH["dup_share"], 0.01)

for src in ["A", "B"]:
    n_unique = train_df.loc[train_df["source"] == src, "calories"].nunique()
    assert_close(f"n_unique_{src}", n_unique, GROUND_TRUTH[f"n_unique_{src}"], 0)

print("\nAll ground-truth assertions PASSED — data on disk matches analysis.md.")
TRAIN_MEDIANS = medians.to_dict()   # {'A': 172.0, 'B': 603.0} -- used later for baselines/clipping sanity
print("TRAIN_MEDIANS:", TRAIN_MEDIANS)

Verifying dataset against analysis.md ground truth...
  [OK] n_train: got=3098.0000 expected=3098 (tol=0)
  [OK] n_test: got=547.0000 expected=547 (tol=0)
  [OK] n_train_A: got=2355.0000 expected=2355 (tol=0)
  [OK] n_train_B: got=743.0000 expected=743 (tol=0)
  [OK] n_test_A: got=417.0000 expected=417 (tol=0)
  [OK] n_test_B: got=130.0000 expected=130 (tol=0)
  [OK] target_min: got=50.0000 expected=50.0 (tol=0.5)
  [OK] target_max: got=3724.1500 expected=3724.15 (tol=0.5)
  [OK] global_mean: got=380.8158 expected=380.8 (tol=1.0)
  [OK] global_median: got=220.0000 expected=220.0 (tol=0.5)
  [OK] skew_raw: got=3.8971 expected=3.9 (tol=0.1)
  [OK] skew_log1p: got=0.6631 expected=0.66 (tol=0.1)
  [OK] median_A: got=172.0000 expected=172.0 (tol=0.5)
  [OK] median_B: got=603.0000 expected=603.0 (tol=0.5)
  [OK] mean_A: got=212.5155 expected=212.5 (tol=1.0)
  [OK] mean_B: got=914.2575 expected=914.3 (tol=1.0)
  [OK] per_source_median_mae: got=214.6193 expected=214.6 (tol=0.5)
  [OK] dup_shar

## 7. Baselines — the bar every model must clear

Per `analysis.md` §5, MAE is minimized by the **conditional median**, so
constant predictors are meaningful baselines, not strawmen. We compute all
four here on the full training set. **The real bar is the per-source median
(~214.6 MAE)** — logged per source too, since that's the number the final
model must beat *for both A and B individually*, not just on average (a model
could "beat" the blended baseline while quietly losing to it on B, which is
the split that matters for the private leaderboard).

We also write a **constant per-source-median submission** as a floor artifact
— if the trained model ever somehow scores worse than this on the leaderboard,
something is broken.

**Runs on: CPU.**

In [ ]:
def mae(y_true, y_pred):
    return np.abs(np.asarray(y_true) - np.asarray(y_pred)).mean()

global_mean, global_median = train_df["calories"].mean(), train_df["calories"].median()
baseline_rows = []
baseline_rows.append(("global mean", mae(train_df["calories"], global_mean)))
baseline_rows.append(("global median", mae(train_df["calories"], global_median)))

per_source_mean_pred = train_df["source"].map(means)
per_source_median_pred = train_df["source"].map(medians)
baseline_rows.append(("per-source mean", mae(train_df["calories"], per_source_mean_pred)))
baseline_rows.append(("per-source median", mae(train_df["calories"], per_source_median_pred)))

print(f"{'baseline':<22}{'MAE':>10}")
for name, val in baseline_rows:
    print(f"{name:<22}{val:>10.2f}")

print("\nPer-source median MAE (this is the number every model must beat, PER SOURCE):")
for src in ["A", "B"]:
    sub = train_df[train_df["source"] == src]
    m = mae(sub["calories"], medians[src])
    print(f"  source {src}: median={medians[src]:.1f} kcal -> in-sample MAE={m:.2f} (n={len(sub)})")

# Floor artifact: constant per-source-median submission
floor_sub = test_df[["image_id"]].copy()
floor_sub["predicted_calories"] = test_df["source"].map(medians)
floor_path = f"{PROJECT_DIR}/submission_floor_median.csv"
floor_sub.to_csv(floor_path, index=False)
print(f"\nWrote floor baseline submission to {floor_path} (per-source median constant, {len(floor_sub)} rows)")
assert len(floor_sub) == 547 and floor_sub["predicted_calories"].isna().sum() == 0

baseline                     MAE
global mean               297.78
global median             255.85
per-source mean           237.32
per-source median         214.62

Per-source median MAE (this is the number every model must beat, PER SOURCE):
  source A: median=172.0 kcal -> in-sample MAE=106.66 (n=2355)
  source B: median=603.0 kcal -> in-sample MAE=556.82 (n=743)

Wrote floor baseline submission to /content/drive/MyDrive/calorie_comp/submission_floor_median.csv (per-source median constant, 547 rows)


## 8. Group-aware, source-stratified CV — the mandatory fix for leakage

`analysis.md` §6: **95.7% of labels repeat** (only 500 unique values across
2,355 A images, only 207 across 743 B images) — a strong signal of the same
physical dish photographed multiple times. If near-duplicate frames land in
both train and validation, CV becomes optimistic and model selection breaks.

**Grouping strategy** (within each source separately — A and B never share
a group):
1. Compute a **perceptual hash (pHash)** per image (resized small first for
   speed) and union any two images within the same source whose Hamming
   distance ≤ `phash_hamming_thresh` (near-duplicate frames of the same
   plate/angle).
2. **Merge** with **exact-calorie-value clusters**: any two images in the same
   source with the identical calorie value are unioned into the same group too
   (catches duplicate dishes photographed from a different angle/frame that
   pHash might miss, per the explicit brief).
3. Connected components of the union give the final `group_id`.

**Stratification**: `source × per-source-calorie-quintile`, so every fold sees
a representative mix of both sources and the full calorie range of each,
while `StratifiedGroupKFold` guarantees no group crosses a fold boundary.

Splits are **saved to `folds.csv`** once and reloaded on every re-run so the CV
is fixed and comparable across experiments.

**Runs on: CPU** (pHash on 3,098 PNG + 743 JPG takes a few minutes; JPGs are
12MP so we downsample via `Image.thumbnail` before hashing).

In [ ]:
import imagehash

def compute_phash_safe(path, hash_size=8):
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img)  # fix phone EXIF rotation before hashing
            img = img.convert("RGB")
            img.thumbnail((256, 256))
            return imagehash.phash(img, hash_size=hash_size)
    except Exception as e:
        warnings.warn(f"pHash failed for {path}: {e} -- using a zero hash (isolated group)")
        return imagehash.ImageHash(np.zeros((hash_size, hash_size), dtype=bool))

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

def build_groups_for_source(df_src: pd.DataFrame, hamming_thresh: int) -> np.ndarray:
    n = len(df_src)
    uf = UnionFind(n)

    # --- pHash near-duplicate clustering ---
    hashes = [compute_phash_safe(p) for p in df_src["path"]]
    bits = np.stack([h.hash.flatten() for h in hashes]).astype(np.uint8)  # (n, 64) bool as 0/1
    from scipy.spatial.distance import pdist, squareform
    ham = squareform(pdist(bits, metric="hamming")) * bits.shape[1]  # back to bit-count distance
    close_pairs = np.argwhere(np.triu(ham <= hamming_thresh, k=1))
    for i, j in close_pairs:
        uf.union(int(i), int(j))

    # --- exact-calorie-value clustering (merge) ---
    cal_round = df_src["calories"].round(2).values
    val_to_idx = defaultdict(list)
    for i, v in enumerate(cal_round):
        val_to_idx[v].append(i)
    for idxs in val_to_idx.values():
        for k in range(1, len(idxs)):
            uf.union(idxs[0], idxs[k])

    roots = np.array([uf.find(i) for i in range(n)])
    # relabel roots to compact 0..k-1 ids (per source)
    _, group_ids = np.unique(roots, return_inverse=True)
    return group_ids, close_pairs.shape[0]

if os.path.exists(FOLDS_CSV):
    print(f"folds.csv already exists at {FOLDS_CSV} -- loading fixed splits (delete the file to force a rebuild).")
    folds_df = pd.read_csv(FOLDS_CSV)
    assert set(folds_df["image_id"]) == set(train_df["image_id"]), \
        "folds.csv image_id set doesn't match current train_labels.csv -- delete folds.csv to rebuild."
    train_df = train_df.merge(folds_df[["image_id", "group_id", "strat_label", "fold"]], on="image_id", how="left")
else:
    print("Building pHash + exact-calorie groups per source (first run only)...")
    group_id_global = np.full(len(train_df), -1, dtype=np.int64)
    offset = 0
    for src in ["A", "B"]:
        mask = (train_df["source"] == src).values
        idx_src = np.where(mask)[0]
        df_src = train_df.iloc[idx_src].reset_index(drop=True)
        gids, n_close = build_groups_for_source(df_src, CFG.phash_hamming_thresh)
        group_id_global[idx_src] = gids + offset
        offset += gids.max() + 1
        print(f"  source {src}: {len(df_src)} images -> {gids.max()+1} groups "
              f"({n_close} near-duplicate pHash pairs found, thresh<={CFG.phash_hamming_thresh})")
    train_df["group_id"] = group_id_global

    # stratification label: source x per-source calorie quintile
    strat_label = pd.Series(index=train_df.index, dtype=object)
    for src in ["A", "B"]:
        mask = train_df["source"] == src
        bins = pd.qcut(train_df.loc[mask, "calories"], q=CFG.n_calorie_bins, labels=False, duplicates="drop")
        strat_label.loc[mask] = [f"{src}_{b}" for b in bins]
    train_df["strat_label"] = strat_label

    from sklearn.model_selection import StratifiedGroupKFold
    sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    train_df["fold"] = -1
    for fold, (_, va_idx) in enumerate(sgkf.split(train_df, y=train_df["strat_label"], groups=train_df["group_id"])):
        train_df.loc[train_df.index[va_idx], "fold"] = fold
    assert (train_df["fold"] >= 0).all()

    train_df[["image_id", "source", "calories", "group_id", "strat_label", "fold"]].to_csv(FOLDS_CSV, index=False)
    print(f"Saved fixed folds to {FOLDS_CSV}")

print(train_df["fold"].value_counts().sort_index())
print(train_df.groupby(["fold", "source"]).size().unstack())

folds.csv already exists at /content/drive/MyDrive/calorie_comp/folds.csv -- loading fixed splits (delete the file to force a rebuild).
fold
0    1120
1     499
2     493
3     492
4     494
Name: count, dtype: int64
source    A    B
fold            
0       973  147
1       350  149
2       345  148
3       341  151
4       346  148


## 9. Quantify leakage optimism: random vs grouped CV

Two checks, both required before trusting any CV number downstream:

1. **Group-overlap leakage stat**: under a naive random `StratifiedKFold`, what
   fraction of validation-fold groups *also appear in that fold's training
   split*? For a truly group-aware split this must be exactly 0%; for random
   splitting, given 95.7% duplication, it should be large — this is the
   concrete mechanism by which random CV lets a model "cheat" (memorize a
   near-duplicate's label instead of learning to predict from pixels).
2. **OOF per-source-median baseline MAE**, computed under both schemes. This
   baseline predicts a constant per source (it can't memorize individual
   dishes), so we *expect* it to be nearly identical under random vs grouped —
   which itself is an important sanity check: it confirms any gap we see later
   between random-CV and grouped-CV *model* performance is coming from the
   model exploiting duplicate leakage, not from some artifact of the split
   mechanics themselves.

**Runs on: CPU.**

In [ ]:
from sklearn.model_selection import StratifiedKFold

def oof_per_source_median_mae(df, fold_col):
    preds = np.zeros(len(df))
    for f in sorted(df[fold_col].unique()):
        tr_mask = df[fold_col] != f
        va_mask = df[fold_col] == f
        fold_medians = df.loc[tr_mask].groupby("source")["calories"].median()
        preds[va_mask.values] = df.loc[va_mask, "source"].map(fold_medians).values
    return mae(df["calories"], preds)

def group_overlap_rate(df, fold_col):
    overlaps = []
    for f in sorted(df[fold_col].unique()):
        tr_groups = set(df.loc[df[fold_col] != f, "group_id"])
        va_groups = df.loc[df[fold_col] == f, "group_id"]
        overlap = va_groups.isin(tr_groups).mean()
        overlaps.append(overlap)
    return np.mean(overlaps)

# --- random (naive) StratifiedKFold, ignoring groups ---
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
random_fold = np.full(len(train_df), -1)
for f, (_, va_idx) in enumerate(skf.split(train_df, train_df["strat_label"])):
    random_fold[va_idx] = f
train_df["fold_random"] = random_fold

random_oof_mae = oof_per_source_median_mae(train_df, "fold_random")
grouped_oof_mae = oof_per_source_median_mae(train_df, "fold")
random_overlap = group_overlap_rate(train_df, "fold_random")
grouped_overlap = group_overlap_rate(train_df, "fold")

print("Per-source-median OOF baseline MAE:")
print(f"  random StratifiedKFold (ignores groups): {random_oof_mae:.2f}")
print(f"  grouped StratifiedGroupKFold (mandatory): {grouped_oof_mae:.2f}")
print(f"  (expected to be close -- constant baselines can't memorize duplicates)\n")

print("Group-overlap leakage rate (fraction of validation rows whose group_id also appears in that fold's training set):")
print(f"  random StratifiedKFold: {random_overlap*100:.1f}%  <-- this is the leakage channel a memorizing model would exploit")
print(f"  grouped StratifiedGroupKFold: {grouped_overlap*100:.1f}%  <-- must be ~0%")
assert grouped_overlap < 0.01, "Grouped CV should have (near-)zero group overlap between train and validation."

train_df = train_df.drop(columns=["fold_random"])

Per-source-median OOF baseline MAE:
  random StratifiedKFold (ignores groups): 214.79
  grouped StratifiedGroupKFold (mandatory): 226.97
  (expected to be close -- constant baselines can't memorize duplicates)

Group-overlap leakage rate (fraction of validation rows whose group_id also appears in that fold's training set):
  random StratifiedKFold: 95.1%  <-- this is the leakage channel a memorizing model would exploit
  grouped StratifiedGroupKFold: 0.0%  <-- must be ~0%


## 10. `SMOKE_TEST` subsetting

If `CFG.SMOKE_TEST` is `True`, we now cut both train and test down to a small
random subset (`smoke_per_source` images per source for train; a proportional
small slice of test) — **after** fold assignment, so fold membership and group
integrity are untouched, just restricted to a subset. Every downstream cell
(cache, dataset, model, training, OOF, inference, submission writer) reads
`train_df`/`test_df` from this point forward, so smoke mode exercises the
exact same code path as a full run, just fast.

A separate cache filename suffix (`_smoke`) is used so smoke and full runs
never clobber each other's Drive cache.

**Runs on: CPU.**

In [ ]:
CACHE_SUFFIX = ""

if CFG.SMOKE_TEST:
    rng = np.random.default_rng(CFG.seed)
    keep_idx = []
    for src in ["A", "B"]:
        src_idx = train_df.index[train_df["source"] == src].to_numpy()
        n_keep = min(CFG.smoke_per_source, len(src_idx))
        keep_idx.append(rng.choice(src_idx, size=n_keep, replace=False))
    keep_idx = np.concatenate(keep_idx)
    train_df = train_df.loc[keep_idx].reset_index(drop=True)

    test_keep_idx = []
    for src in ["A", "B"]:
        src_idx = test_df.index[test_df["source"] == src].to_numpy()
        n_keep = max(5, int(len(src_idx) * (CFG.smoke_per_source * 2 / 3098)))
        n_keep = min(n_keep, len(src_idx))
        test_keep_idx.append(rng.choice(src_idx, size=n_keep, replace=False))
    test_df = test_df.loc[np.sort(np.concatenate(test_keep_idx))].reset_index(drop=True)  # sorted to preserve original test_ids.csv relative order

    CACHE_SUFFIX = "_smoke"
    print(f"SMOKE_TEST active: train subset n={len(train_df)}, test subset n={len(test_df)}")
    print(train_df.groupby(["fold", "source"]).size().unstack(fill_value=0))
else:
    print(f"Full run: train n={len(train_df)}, test n={len(test_df)}")

Full run: train n=3098, test n=547


## 11. Build the letterbox image cache (once, on Drive)

Reading thousands of small files from Drive every epoch is slow, and Source
B's 12.2MP JPEGs are expensive to decode repeatedly. So we **decode + letterbox
resize once**, stack everything into a couple of big `.npy` arrays, and save
those to Drive. On every subsequent run we just check whether a valid cache
already exists (same `image_id` set, same `IMG_SIZE`) and skip straight to
loading if so.

**Letterbox, not crop or squash**: `analysis.md` explicitly warns that a
center-crop can cut off food that occupies little of the frame (A: small item
on a big plate; B: plate in a large dark margin), and squashing to a fixed
aspect ratio distorts apparent portion size — which is the actual signal this
whole task depends on. So we aspect-preserving-resize the long side to
`IMG_SIZE` and pad the short side with black to a square, for **both sources**.

Defensive loading: `PIL.ImageOps.exif_transpose` fixes phone EXIF rotation tags
(Source B has both portrait 3024×4032 and landscape 4032×3024 raw files, and
without this some "portrait" JPEGs would decode sideways), and any decode
failure falls back to a mid-gray image with a loud warning rather than
crashing the whole cache build.

**Runs on: CPU** (no GPU needed; this is pure image I/O + resize, but it's the
long pole time-wise on a full run — expect several minutes for 3,098 + 547
images at 384px, mostly from decoding Source B's 12MP JPEGs).

In [ ]:
def letterbox_resize(img: np.ndarray, size: int, pad_value: int = 0) -> np.ndarray:
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(img, (nw, nh), interpolation=interp)
    canvas = np.full((size, size, 3), pad_value, dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas

def load_image_defensive(path: str, size: int) -> np.ndarray:
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img)  # handles B portrait/landscape EXIF rotation
            img = img.convert("RGB")
            arr = np.array(img)
        return letterbox_resize(arr, size)
    except Exception as e:
        warnings.warn(f"Defensive load failed for {path}: {e} -- using mid-gray placeholder")
        return np.full((size, size, 3), 128, dtype=np.uint8)

def build_or_load_cache(df: pd.DataFrame, split_name: str, img_size: int, cache_dir: str, suffix: str):
    arr_path = f"{cache_dir}/{split_name}_cache_{img_size}{suffix}.npy"
    idx_path = f"{cache_dir}/{split_name}_cache_{img_size}{suffix}_index.csv"

    if os.path.exists(arr_path) and os.path.exists(idx_path):
        idx_df = pd.read_csv(idx_path)
        if list(idx_df["image_id"]) == list(df["image_id"]):
            print(f"[{split_name}] valid cache found at {arr_path} (n={len(idx_df)}) -- skipping rebuild.")
            return arr_path, idx_path
        else:
            print(f"[{split_name}] cache at {arr_path} exists but image_id order/set differs from current df -- rebuilding.")

    print(f"[{split_name}] building letterbox cache at size {img_size} for {len(df)} images...")
    t0 = time.time()
    arr = np.empty((len(df), img_size, img_size, 3), dtype=np.uint8)
    for i, path in enumerate(df["path"].values):
        arr[i] = load_image_defensive(path, img_size)
        if (i + 1) % 500 == 0 or (i + 1) == len(df):
            print(f"  {i+1}/{len(df)} ({time.time()-t0:.1f}s)")
    np.save(arr_path, arr)
    df[["image_id"]].to_csv(idx_path, index=False)
    print(f"[{split_name}] cache built and saved to {arr_path} ({arr.nbytes/1e9:.2f} GB), took {time.time()-t0:.1f}s")
    return arr_path, idx_path

train_arr_path, train_idx_path = build_or_load_cache(train_df, "train", CFG.img_size, CACHE_DIR, CACHE_SUFFIX)
test_arr_path, test_idx_path = build_or_load_cache(test_df, "test", CFG.img_size, CACHE_DIR, CACHE_SUFFIX)

[train] valid cache found at /content/drive/MyDrive/calorie_comp/cache/train_cache_384.npy (n=3098) -- skipping rebuild.
[test] valid cache found at /content/drive/MyDrive/calorie_comp/cache/test_cache_384.npy (n=547) -- skipping rebuild.


## 12. Copy the cache to local disk for fast epoch I/O

Drive-mounted reads are network calls under the hood; re-reading the cache
array from Drive on every batch would bottleneck training. We copy the two
`.npy` files to local `/content` (Colab's local SSD) once per session and load
from there for the rest of the notebook — checkpoints, `folds.csv`, weights,
and the final submission still live on Drive so nothing is lost if the local
disk is wiped on disconnect.

**Runs on: CPU.**

In [ ]:
def copy_to_local(src_path: str, local_dir: str) -> str:
    dst_path = f"{local_dir}/{os.path.basename(src_path)}"
    if os.path.exists(dst_path) and os.path.getsize(dst_path) == os.path.getsize(src_path):
        print(f"already local: {dst_path}")
        return dst_path
    t0 = time.time()
    shutil.copy(src_path, dst_path)
    print(f"copied {src_path} -> {dst_path} ({time.time()-t0:.1f}s)")
    return dst_path

train_arr_local = copy_to_local(train_arr_path, LOCAL_CACHE_DIR)
test_arr_local = copy_to_local(test_arr_path, LOCAL_CACHE_DIR)

train_cache = np.load(train_arr_local, mmap_mode=None)   # small enough to hold in RAM
test_cache = np.load(test_arr_local, mmap_mode=None)
assert train_cache.shape[0] == len(train_df) and test_cache.shape[0] == len(test_df)
print("train_cache:", train_cache.shape, train_cache.dtype, "| test_cache:", test_cache.shape, test_cache.dtype)

already local: /content/local_cache/train_cache_384.npy
already local: /content/local_cache/test_cache_384.npy
train_cache: (3098, 384, 384, 3) uint8 | test_cache: (547, 384, 384, 3) uint8


## 13. Portion-safe augmentations (source-specific)

Per `analysis.md` §7, calories are read off *apparent portion*, so any
augmentation that changes how much food appears to be in frame is actively
harmful — this rules out `RandomResizedCrop`/aggressive zoom, `mixup`,
`cutmix`, and heavy cutout, all of which either crop away food, blend two
different portions together, or occlude large chunks of the plate.

What's safe and what's applied:

| | Source A (top-down) | Source B (oblique) |
|---|---|---|
| hflip | ✅ p=0.5 | ✅ p=0.5 |
| vflip | ✅ p=0.5 (top-down view has no "up") | ❌ (would flip gravity/perspective cues) |
| rotation | ✅ random 90°/180°/270° (view is rotation-invariant from directly above) | ✅ small ±10° only (mild camera tilt, not a full spin) |
| color jitter | ✅ mild (brightness/contrast/saturation/hue) — helps bridge the lab-rig vs smartphone domain gap | ✅ mild |
| scale jitter | ✅ mild 0.9–1.1× (small zoom in/out, not a crop) | ✅ mild 0.9–1.1× |
| RandomResizedCrop / mixup / cutmix / heavy cutout | ❌ forbidden | ❌ forbidden |

Scale jitter is implemented as a resize-then-pad-or-center-crop *of the already
square, letterboxed 384×384 canvas* by ±10% — this changes apparent zoom only
slightly and never discards more than a thin border, unlike `RandomResizedCrop`
which can crop out half the plate.

Normalization uses standard ImageNet mean/std (required for the pretrained
ConvNeXt backbone).

**Runs on: CPU** (augmentation happens in the `Dataset.__getitem__`, executed
by CPU `DataLoader` workers; only the resulting tensors go to GPU).

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
SOURCE_TO_IDX = {"A": 0, "B": 1}

def hflip(img):
    return img[:, ::-1, :]

def vflip(img):
    return img[::-1, :, :]

def rot90k(img, k):
    return np.rot90(img, k=k, axes=(0, 1))

def small_rotate(img, max_deg, rng: np.random.Generator):
    deg = rng.uniform(-max_deg, max_deg)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), deg, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT, borderValue=(CFG.pad_value,) * 3)

def scale_jitter(img, lo, hi, rng: np.random.Generator):
    size = img.shape[0]
    s = rng.uniform(lo, hi)
    new_size = max(1, int(round(size * s)))
    resized = cv2.resize(img, (new_size, new_size),
                          interpolation=cv2.INTER_AREA if s < 1.0 else cv2.INTER_LINEAR)
    if new_size == size:
        return resized
    if new_size < size:  # zoomed out -> pad back to size, centered
        canvas = np.full((size, size, 3), CFG.pad_value, dtype=np.uint8)
        off = (size - new_size) // 2
        canvas[off:off + new_size, off:off + new_size] = resized
        return canvas
    off = (new_size - size) // 2  # zoomed in -> center-crop back to size
    return resized[off:off + size, off:off + size]

def color_jitter(img, rng: np.random.Generator):
    img = img.astype(np.float32)
    brightness = rng.uniform(0.85, 1.15)
    contrast = rng.uniform(0.85, 1.15)
    saturation = rng.uniform(0.9, 1.1)
    img = img * brightness
    mean_gray = img.mean()
    img = (img - mean_gray) * contrast + mean_gray
    gray = img.mean(axis=2, keepdims=True)
    img = (img - gray) * saturation + gray
    return np.clip(img, 0, 255).astype(np.uint8)

def augment(img: np.ndarray, source: str, rng: np.random.Generator) -> np.ndarray:
    if rng.random() < 0.5:
        img = hflip(img)
    if source == "A":
        if rng.random() < 0.5:
            img = vflip(img)
        k = int(rng.integers(0, 4))
        if k:
            img = rot90k(img, k)
    else:  # source B
        img = small_rotate(img, max_deg=10.0, rng=rng)
    img = color_jitter(img, rng)
    img = scale_jitter(img, 0.9, 1.1, rng)
    return np.ascontiguousarray(img)

def to_tensor_normalized(img: np.ndarray) -> torch.Tensor:
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(img.transpose(2, 0, 1)).float()

class CalorieDataset(Dataset):
    def __init__(self, cache_array: np.ndarray, df: pd.DataFrame, train: bool, seed: int = 0):
        self.cache = cache_array
        self.df = df.reset_index(drop=True)
        self.train = train
        self.has_labels = "calories" in self.df.columns
        self.base_seed = seed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        img = self.cache[i]
        source = self.df.at[i, "source"]
        if self.train:
            worker_info = torch.utils.data.get_worker_info()
            worker_seed = (worker_info.seed if worker_info is not None else torch.initial_seed())
            rng = np.random.default_rng((worker_seed + i) % (2**32))
            img = augment(img, source, rng)
        x = to_tensor_normalized(img)
        source_idx = SOURCE_TO_IDX[source]
        if self.has_labels:
            y_kcal = float(self.df.at[i, "calories"])
            return x, torch.tensor(source_idx, dtype=torch.long), torch.tensor(y_kcal, dtype=torch.float32)
        return x, torch.tensor(source_idx, dtype=torch.long), self.df.at[i, "image_id"]

def make_loader(cache_array, df, train: bool, batch_size: int, seed: int):
    ds = CalorieDataset(cache_array, df, train=train, seed=seed)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=train, drop_last=False,
        num_workers=CFG.num_workers, worker_init_fn=seed_worker,
        generator=make_generator(seed), pin_memory=torch.cuda.is_available(),
    )

print("Dataset/DataLoader utilities defined.")

Dataset/DataLoader utilities defined.


## 14. Backbone weights — download once, load offline thereafter

`timm`'s `convnext_tiny` ImageNet-1k weights are downloaded **once** (internet
is on in Colab) and the raw `state_dict` is saved to
`PROJECT_DIR/weights/convnext_tiny_in1k.pth`. Every subsequent run — including
after a Colab disconnect — loads from that Drive file with
`timm.create_model(pretrained=False)` + `load_state_dict`, so training never
depends on the download succeeding again. (ImageNet-pretrained weights are
allowed per the rules; they are not "external data" for this task, just a
standard pretrained backbone.)

**Runs on: CPU** (download only; no forward/backward pass here).

In [ ]:
BACKBONE_WEIGHTS_PATH = f"{WEIGHTS_DIR}/{CFG.backbone_name}_in1k.pth"

import timm

if not os.path.exists(BACKBONE_WEIGHTS_PATH):
    print(f"Downloading pretrained {CFG.backbone_name} (ImageNet-1k) -- one-time only...")
    _tmp_model = timm.create_model(CFG.backbone_name, pretrained=True)
    torch.save(_tmp_model.state_dict(), BACKBONE_WEIGHTS_PATH)
    del _tmp_model
    print(f"Saved backbone weights to {BACKBONE_WEIGHTS_PATH}")
else:
    print(f"Backbone weights already cached at {BACKBONE_WEIGHTS_PATH} -- skipping download.")

Backbone weights already cached at /content/drive/MyDrive/calorie_comp/weights/convnext_tiny_in1k.pth -- skipping download.


## 15. Model — shared ConvNeXt-Tiny backbone + source embedding + MLP head

Per `analysis.md` §7, the two sources are **known at inference** (extension →
source is deterministic and identical between train/test), so it's free
information the model should condition on rather than trying to infer. The
design here is the single shared-backbone variant from the brief:

`image → ConvNeXt-Tiny (shared, ImageNet-pretrained) → global-avg-pool feature
→ concat with a learned source embedding (dim 16, keyed on A/B) → small MLP
head → one scalar in log1p(kcal) space`

A single shared backbone (rather than two separate backbones/heads per source)
lets Source A's much larger sample count (76%) help Source B learn general
food-texture representations, while the source embedding gives the head an
explicit, cheap way to shift its output distribution per source (recall A
medians ~172, B medians ~603 — a ~3.5× gap) without needing enough B-only data
to learn that shift purely from pixels.

**Runs on: CPU (definition) / GPU (training, later cells).**

In [ ]:
class CalorieNet(nn.Module):
    def __init__(self, backbone_name: str, weights_path: str, source_emb_dim: int,
                 head_hidden: int, head_dropout: float, n_sources: int = 2):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, global_pool="avg")
        state_dict = torch.load(weights_path, map_location="cpu")
        missing, unexpected = self.backbone.load_state_dict(state_dict, strict=False)
        # expected: 'head.fc.*' unexpected (we dropped the 1000-way classifier); nothing else should be missing
        bad_missing = [m for m in missing if not m.startswith("head.")]
        assert not bad_missing, f"Unexpected missing backbone keys when loading pretrained weights: {bad_missing}"

        feat_dim = self.backbone.num_features
        self.source_emb = nn.Embedding(n_sources, source_emb_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim + source_emb_dim, head_hidden),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x, source_idx):
        feat = self.backbone(x)
        emb = self.source_emb(source_idx)
        z = self.head(torch.cat([feat, emb], dim=1)).squeeze(1)  # log1p(kcal) space
        return z

def build_model():
    return CalorieNet(
        CFG.backbone_name, BACKBONE_WEIGHTS_PATH, CFG.source_emb_dim,
        CFG.head_hidden, CFG.head_dropout,
    )

_test_model = build_model()
n_params = sum(p.numel() for p in _test_model.parameters())
print(f"Model built OK: {CFG.backbone_name} + source-embed({CFG.source_emb_dim}) + MLP head. "
      f"Total params: {n_params/1e6:.1f}M")
del _test_model

Model built OK: convnext_tiny + source-embed(16) + MLP head. Total params: 28.0M


## 16. Loss — L1 in **kcal space**, computed from a log1p-space prediction

The network outputs a scalar `z` meant to approximate `log1p(kcal)` (fixes the
3.90 raw skew per `analysis.md` §4). But MAE is the competition metric **in
kcal space**, and a naive L1-on-`log1p` loss would implicitly weight a
proportional log-error the same for a 100 kcal salad and a 3000 kcal feast —
exactly backwards from what kcal-space MAE rewards, and exactly why Source B
(the high-kcal, high-variance regime) needs to be up-weighted rather than
down-weighted. So the loss **inverts the transform before comparing**:

`loss = |expm1(clamp(z)) - y_kcal|`

which makes gradients scale with predicted kcal, naturally up-weighting Source
B's larger errors — matching the metric we're actually optimizing for. `z` is
clamped before `expm1` (at `log1p(4000)`, comfortably above the observed
max of 3724) purely to prevent a single bad early-training prediction from
`exp`-exploding into `inf`/`nan`; it is not a prediction clip (that happens
separately, per-source, at inference time in §21).

A Huber(δ≈100 kcal) version of the same kcal-space construction is included as
a **stability fallback** (`CFG.loss_type = "huber"`) in case raw L1 proves
unstable early in training — Huber is quadratic (smoother gradient) inside
±100 kcal of the target and linear (same as L1) beyond it.
Gradient-norm clipping at 1.0 is applied on top of either loss as a further
stabilizer (see the training step in §18).

**Runs on: CPU (definition) / GPU (used during training).**

In [ ]:
LOG1P_CLAMP_MAX = math.log1p(4000.0)  # safety clamp, not a prediction clip -- see §21 for real clipping

def kcal_l1_loss(z_pred, y_kcal):
    z = torch.clamp(z_pred, max=LOG1P_CLAMP_MAX)
    pred_kcal = torch.expm1(z)
    return (pred_kcal - y_kcal).abs().mean()

def kcal_huber_loss(z_pred, y_kcal, delta=100.0):
    z = torch.clamp(z_pred, max=LOG1P_CLAMP_MAX)
    pred_kcal = torch.expm1(z)
    diff = (pred_kcal - y_kcal).abs()
    quad = torch.clamp(diff, max=delta)
    lin = diff - quad
    return (0.5 * quad**2 + delta * lin).mean()

def compute_loss(z_pred, y_kcal):
    if CFG.loss_type == "l1":
        return kcal_l1_loss(z_pred, y_kcal)
    elif CFG.loss_type == "huber":
        return kcal_huber_loss(z_pred, y_kcal, delta=CFG.huber_delta)
    raise ValueError(f"Unknown loss_type: {CFG.loss_type}")

print(f"Loss configured: {CFG.loss_type} (kcal-space), clamp z at log1p(4000)={LOG1P_CLAMP_MAX:.3f}")

Loss configured: l1 (kcal-space), clamp z at log1p(4000)=8.294


## 17. Optimizer & schedule

`AdamW` with **differential learning rates** — head (source embedding + MLP,
freshly initialized) at `1e-3`, backbone (pretrained, needs gentler updates)
at `1e-4` — weight decay `0.05`. Cosine decay with a 1-epoch linear warmup,
stepped every optimizer step (i.e. every `accum_steps` micro-batches, not every
micro-batch) so the schedule matches the actual number of gradient updates
regardless of GPU-driven micro-batch size.

**Runs on: CPU (definition) / GPU (used during training).**

In [ ]:
def build_optimizer(model):
    return torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CFG.backbone_lr},
        {"params": list(model.source_emb.parameters()) + list(model.head.parameters()), "lr": CFG.head_lr},
    ], weight_decay=CFG.weight_decay)

def build_scheduler(optimizer, steps_per_epoch: int, epochs: int, warmup_epochs: int):
    total_steps = max(1, steps_per_epoch * epochs)
    warmup_steps = steps_per_epoch * warmup_epochs
    def lr_lambda(step):
        if warmup_steps > 0 and step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("Optimizer/scheduler builders defined.")

Optimizer/scheduler builders defined.


## 18. Training step, validation step, and resumable checkpointing

- **Gradient accumulation** keeps the *effective* batch size fixed at
  `CFG.effective_batch` regardless of which GPU is detected: micro-batches of
  `GPU_INFO["micro_batch"]` are accumulated for `accum_steps` iterations before
  each optimizer step, so an A100 (large micro-batch, few accum steps) and a
  T4 (small micro-batch, many accum steps) run the *same* optimization
  trajectory.
- **Mixed precision**: `bf16` autocast on A100 (no `GradScaler` needed — bf16
  has enough dynamic range), `fp16` + `GradScaler` on T4/L4.
- **Grad-clip norm 1.0** applied after unscaling, before the optimizer step.
- **Checkpoint every epoch** (`fold{f}_last.pt`: model/optimizer/scheduler/RNG
  state + epoch number) so a Colab disconnect loses at most one epoch. A
  separate `fold{f}_best.pt` is written whenever validation MAE improves — that
  file is what gets used later for OOF collection and final inference.
- **Per-source validation MAE** is reported every epoch (not just overall),
  since Source B dominates total MAE and could regress while overall MAE still
  looks fine.

**Runs on: GPU** (training/validation forward+backward passes); the
checkpoint I/O itself is CPU-only.

In [ ]:
ACCUM_STEPS = max(1, round(CFG.effective_batch / GPU_INFO["micro_batch"]))
print(f"Micro-batch={GPU_INFO['micro_batch']}, accum_steps={ACCUM_STEPS} "
      f"-> effective batch={GPU_INFO['micro_batch']*ACCUM_STEPS} (target {CFG.effective_batch})")

def train_one_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss, n_seen = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, (x, source_idx, y_kcal) in enumerate(loader):
        x, source_idx, y_kcal = x.to(device, non_blocking=True), source_idx.to(device, non_blocking=True), y_kcal.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                             dtype=GPU_INFO["amp_dtype"], enabled=device.type == "cuda"):
            z_pred = model(x, source_idx)
            loss = compute_loss(z_pred, y_kcal) / ACCUM_STEPS

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        is_last = (step + 1) == len(loader)
        if (step + 1) % ACCUM_STEPS == 0 or is_last:
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        total_loss += loss.item() * ACCUM_STEPS * x.size(0)
        n_seen += x.size(0)
    return total_loss / max(1, n_seen)

@torch.no_grad()
def validate_one_epoch(model, loader, device, df_for_source):
    model.eval()
    all_preds, all_true, all_src = [], [], []
    for x, source_idx, y_kcal in loader:
        x, source_idx = x.to(device, non_blocking=True), source_idx.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                             dtype=GPU_INFO["amp_dtype"], enabled=device.type == "cuda"):
            z_pred = model(x, source_idx)
        pred_kcal = torch.expm1(torch.clamp(z_pred, max=LOG1P_CLAMP_MAX)).float().cpu().numpy()
        all_preds.append(pred_kcal)
        all_true.append(y_kcal.numpy())
        all_src.append(source_idx.cpu().numpy())
    preds = np.concatenate(all_preds)
    trues = np.concatenate(all_true)
    srcs = np.concatenate(all_src)
    overall_mae = mae(trues, preds)
    per_source_mae = {}
    for src, idx in SOURCE_TO_IDX.items():
        m = srcs == idx
        if m.sum() > 0:
            per_source_mae[src] = mae(trues[m], preds[m])
    return overall_mae, per_source_mae, preds

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_mae):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_mae": best_val_mae,
        "torch_rng": torch.get_rng_state(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, restore_rng=True):
    # weights_only=False: this is our own checkpoint (model/optimizer/scheduler/RNG state), not
    # untrusted third-party data -- PyTorch >=2.6 defaults torch.load to weights_only=True, which
    # rejects the numpy RNG state tuples we store here.
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if restore_rng:
        torch.set_rng_state(ckpt["torch_rng"])
        np.random.set_state(ckpt["numpy_rng"])
        random.setstate(ckpt["python_rng"])
        if ckpt.get("cuda_rng") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(ckpt["cuda_rng"])
    return ckpt["epoch"], ckpt["best_val_mae"]

print("Training/validation/checkpoint utilities defined.")

Micro-batch=64, accum_steps=1 -> effective batch=64 (target 32)
Training/validation/checkpoint utilities defined.


## 19. Train all folds (resumable) — the main training loop

Trains `CFG.n_folds_to_run` folds (1 in `SMOKE_TEST`, else all 5 from
`folds.csv`). For each fold:

1. If `fold{f}_last.pt` already exists on Drive, **resume** from the saved
   epoch/optimizer/scheduler/RNG state instead of starting over — this is what
   makes the notebook robust to Colab disconnects mid-fold.
2. Otherwise start fresh from the offline pretrained backbone.
3. Every epoch: train, validate (overall + per-source MAE), checkpoint
   `_last`, and checkpoint `_best` whenever validation MAE improves.
4. After the fold's epochs are done, reload `_best` and record its
   (uncalibrated) validation predictions into the OOF table — these feed
   §20's per-source calibration fit.

Results are also appended to a `cv_results.csv` log under `LOG_DIR` so the
final mean±std summary in §22 doesn't depend on keeping all fold output in
notebook memory.

**Runs on: GPU** (this is the actual model training — the one cell in the
notebook that meaningfully needs it; expect it to be slow/pointless on CPU
except for `SMOKE_TEST` correctness checking).

In [ ]:
device = GPU_INFO["device"]
oof_records = []
cv_summary = []

folds_to_run = sorted(train_df["fold"].unique())[:CFG.n_folds_to_run]
print(f"Training folds: {folds_to_run} (SMOKE_TEST={CFG.SMOKE_TEST})")

for fold in folds_to_run:
    print(f"\n{'='*60}\nFOLD {fold}\n{'='*60}")
    tr_df = train_df[train_df["fold"] != fold].reset_index(drop=True)
    va_df = train_df[train_df["fold"] == fold].reset_index(drop=True)
    tr_cache = train_cache[(train_df["fold"] != fold).values]
    va_cache = train_cache[(train_df["fold"] == fold).values]

    train_loader = make_loader(tr_cache, tr_df, train=True, batch_size=GPU_INFO["micro_batch"], seed=CFG.seed + fold)
    val_loader = make_loader(va_cache, va_df, train=False, batch_size=GPU_INFO["micro_batch"] * 2, seed=CFG.seed + fold)

    model = build_model().to(device)
    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer, steps_per_epoch=math.ceil(len(train_loader) / ACCUM_STEPS), epochs=CFG.epochs, warmup_epochs=CFG.warmup_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=GPU_INFO["use_grad_scaler"])

    last_ckpt_path = f"{CKPT_DIR}/fold{fold}_last.pt"
    best_ckpt_path = f"{CKPT_DIR}/fold{fold}_best.pt"

    start_epoch, best_val_mae = 0, float("inf")
    if os.path.exists(last_ckpt_path):
        resumed_epoch, best_val_mae = load_checkpoint(last_ckpt_path, model, optimizer, scheduler)
        start_epoch = resumed_epoch + 1
        print(f"Resuming fold {fold} from epoch {start_epoch} (best_val_mae so far={best_val_mae:.2f})")

    for epoch in range(start_epoch, CFG.epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, device)
        val_mae, val_mae_per_source, _ = validate_one_epoch(model, val_loader, device, va_df)
        dt = time.time() - t0

        per_src_str = " ".join(f"MAE_{s}={v:.1f}" for s, v in val_mae_per_source.items())
        print(f"  epoch {epoch+1}/{CFG.epochs} | train_loss={train_loss:.2f} | val_MAE={val_mae:.2f} | {per_src_str} | {dt:.1f}s")

        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, epoch, min(best_val_mae, val_mae))
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            save_checkpoint(best_ckpt_path, model, optimizer, scheduler, epoch, best_val_mae)
            print(f"    -> new best (val_MAE={best_val_mae:.2f}), saved {best_ckpt_path}")

        cv_summary.append({"fold": fold, "epoch": epoch, "train_loss": train_loss, "val_mae": val_mae, **{f"val_mae_{s}": v for s, v in val_mae_per_source.items()}})
        pd.DataFrame(cv_summary).to_csv(f"{LOG_DIR}/cv_results.csv", index=False)

    # reload best checkpoint for this fold and record uncalibrated OOF predictions
    load_checkpoint(best_ckpt_path, model, restore_rng=False)
    _, final_per_source_mae, oof_preds = validate_one_epoch(model, val_loader, device, va_df)
    print(f"  FOLD {fold} BEST -> " + " ".join(f"MAE_{s}={v:.2f}" for s, v in final_per_source_mae.items()))

    fold_oof = va_df[["image_id", "source", "calories"]].copy()
    fold_oof["fold"] = fold
    fold_oof["pred_raw"] = oof_preds
    oof_records.append(fold_oof)

    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

oof_df = pd.concat(oof_records, ignore_index=True)
oof_df.to_csv(f"{OOF_DIR}/oof_predictions.csv", index=False)
print(f"\nSaved OOF predictions for {len(oof_df)} rows to {OOF_DIR}/oof_predictions.csv")

Training folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (SMOKE_TEST=False)

FOLD 0
Resuming fold 0 from epoch 50 (best_val_mae so far=43.82)
  epoch 51/100 | train_loss=108.68 | val_MAE=51.56 | MAE_A=31.1 MAE_B=186.7 | 16.6s
  epoch 52/100 | train_loss=104.15 | val_MAE=54.73 | MAE_A=35.2 MAE_B=183.9 | 16.5s
  epoch 53/100 | train_loss=115.36 | val_MAE=62.03 | MAE_A=32.7 MAE_B=256.0 | 16.8s
  epoch 54/100 | train_loss=112.51 | val_MAE=51.52 | MAE_A=30.9 MAE_B=188.0 | 16.6s
  epoch 55/100 | train_loss=126.48 | val_MAE=64.48 | MAE_A=33.4 MAE_B=270.3 | 16.5s
  epoch 56/100 | train_loss=105.66 | val_MAE=51.65 | MAE_A=30.0 MAE_B=195.2 | 16.9s
  epoch 57/100 | train_loss=95.46 | val_MAE=45.91 | MAE_A=27.1 MAE_B=170.6 | 16.6s
  epoch 58/100 | train_loss=103.83 | val_MAE=74.07 | MAE_A=41.8 MAE_B=287.4 | 16.4s
  epoch 59/100 | train_loss=109.26 | val_MAE=56.85 | MAE_A=31.3 MAE_B=226.1 | 16.6s
  epoch 60/100 | train_loss=106.44 | val_MAE=72.71 | MAE_A=35.5 MAE_B=319.1 | 

## 20. CV summary — overall and per-source, mean ± std across folds

The number that matters: does the trained model clearly beat the ~214.6
per-source-median baseline **on both sources**, not just in aggregate?

**Runs on: CPU** (just aggregates the OOF table from the previous cell).

In [ ]:
overall_mae_oof = mae(oof_df["calories"], oof_df["pred_raw"])
per_source_oof_mae = oof_df.groupby("source").apply(lambda g: mae(g["calories"], g["pred_raw"]))

fold_maes = oof_df.groupby("fold").apply(lambda g: mae(g["calories"], g["pred_raw"]))
print(f"OOF overall MAE: {overall_mae_oof:.2f}  (mean over folds: {fold_maes.mean():.2f} +/- {fold_maes.std():.2f})")
for src in ["A", "B"]:
    if src in per_source_oof_mae.index:
        fold_src_maes = oof_df[oof_df["source"] == src].groupby("fold").apply(lambda g: mae(g["calories"], g["pred_raw"]))
        print(f"OOF MAE source {src}: {per_source_oof_mae[src]:.2f}  "
              f"(mean over folds: {fold_src_maes.mean():.2f} +/- {fold_src_maes.std():.2f}) "
              f"vs baseline {mae(train_df.loc[train_df.source==src,'calories'], TRAIN_MEDIANS[src]):.2f}")

OOF overall MAE: 54.83  (mean over folds: 58.58 +/- 10.71)
OOF MAE source A: 29.96  (mean over folds: 31.96 +/- 5.49) vs baseline 106.66
OOF MAE source B: 133.65  (mean over folds: 133.72 +/- 14.22) vs baseline 556.82


/tmp/ipykernel_1004/2838689411.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_source_oof_mae = oof_df.groupby("source").apply(lambda g: mae(g["calories"], g["pred_raw"]))
/tmp/ipykernel_1004/2838689411.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_maes = oof_df.groupby("fold").apply(lambda g: mae(g["calories"], g["pred_raw"]))
/tmp/ipykernel_1004/2838689411.py:8: DeprecationWarning: Data

## 21. Post-hoc per-source calibration — fit on OOF only, cross-validated selection

Even with the source embedding, the head may systematically over/under-shoot
one source (e.g. underfit Source B's long tail). Three candidate calibrations
are compared **per source**, fit only on OOF predictions (never on test):

- **identity** — no change.
- **additive median-shift** — `pred + median(y_true - pred)` (robust to
  outliers, corrects a systematic offset).
- **affine** — `a * pred + b` via least squares (corrects both offset and
  scale, e.g. if the model compresses Source B's range toward the mean).

**Selection is cross-validated**, not fit-and-evaluate-on-the-same-data: for
each source, each method's params are fit on 4 folds' OOF and evaluated on the
held-out 5th fold's OOF, repeated across all folds, giving an honest
`cv_mae ± fold_std` per method. A non-identity calibration is kept **only if
it beats identity's cv_mae by more than identity's own fold std** — otherwise
we're just fitting noise, and identity (i.e. trust the model's raw output) is
safer. Whichever method is selected per source, its final production
parameters are then refit on the **full** OOF (all folds) for use at test
time in §23.

**Runs on: CPU.**

In [ ]:
def fit_shift(preds, trues):
    return {"shift": float(np.median(trues - preds))}

def apply_shift(preds, params):
    return preds + params["shift"]

def fit_affine(preds, trues):
    a, b = np.polyfit(preds, trues, 1)
    return {"a": float(a), "b": float(b)}

def apply_affine(preds, params):
    return params["a"] * preds + params["b"]

CALIB_METHODS = {
    "identity": (lambda p, t: {}, lambda p, params: p),
    "shift": (fit_shift, apply_shift),
    "affine": (fit_affine, apply_affine),
}

selected_calibration = {}
calibration_report = []

for src in ["A", "B"]:
    src_oof = oof_df[oof_df["source"] == src]
    folds_present = sorted(src_oof["fold"].unique())

    if len(folds_present) < 2:
        # SMOKE_TEST runs a single fold -- there's no "other folds" data left to fit a
        # calibration on, so cross-validated selection is undefined. Default to identity
        # (safe choice) rather than fitting/evaluating on the same data.
        id_mean = mae(src_oof["calories"], src_oof["pred_raw"])
        print(f"source {src}: only {len(folds_present)} fold available (SMOKE_TEST) -- "
              f"cannot cross-validate calibration, defaulting to 'identity' (in-sample MAE={id_mean:.2f}).")
        selected_calibration[src] = {"method": "identity", "params": {}, "apply_fn": CALIB_METHODS["identity"][1]}
        calibration_report.append({"source": src, "method": "identity", "cv_mae": id_mean, "cv_std": float("nan"), "beats_identity_by_more_than_std": None})
        continue

    method_fold_maes = {m: [] for m in CALIB_METHODS}

    for f in folds_present:
        fit_mask = src_oof["fold"] != f
        eval_mask = src_oof["fold"] == f
        fit_preds, fit_trues = src_oof.loc[fit_mask, "pred_raw"].values, src_oof.loc[fit_mask, "calories"].values
        eval_preds, eval_trues = src_oof.loc[eval_mask, "pred_raw"].values, src_oof.loc[eval_mask, "calories"].values
        for name, (fit_fn, apply_fn) in CALIB_METHODS.items():
            params = fit_fn(fit_preds, fit_trues)
            calibrated = apply_fn(eval_preds, params)
            method_fold_maes[name].append(mae(eval_trues, calibrated))

    id_mean, id_std = np.mean(method_fold_maes["identity"]), np.std(method_fold_maes["identity"])
    best_method, best_mean = "identity", id_mean
    for name in ["shift", "affine"]:
        cand_mean = np.mean(method_fold_maes[name])
        improves_enough = (id_mean - cand_mean) > id_std
        calibration_report.append({"source": src, "method": name, "cv_mae": cand_mean,
                                    "cv_std": np.std(method_fold_maes[name]),
                                    "beats_identity_by_more_than_std": improves_enough})
        if improves_enough and cand_mean < best_mean:
            best_method, best_mean = name, cand_mean
    calibration_report.append({"source": src, "method": "identity", "cv_mae": id_mean, "cv_std": id_std, "beats_identity_by_more_than_std": None})

    # refit the selected method's production params on the FULL OOF for this source
    fit_fn, apply_fn = CALIB_METHODS[best_method]
    prod_params = fit_fn(src_oof["pred_raw"].values, src_oof["calories"].values)
    selected_calibration[src] = {"method": best_method, "params": prod_params, "apply_fn": apply_fn}
    print(f"source {src}: selected calibration = '{best_method}' (id_cv_mae={id_mean:.2f}+/-{id_std:.2f}, "
          f"chosen_cv_mae={best_mean:.2f}), production params={prod_params}")

pd.DataFrame(calibration_report).to_csv(f"{OOF_DIR}/calibration_report.csv", index=False)
print(f"\nSaved calibration comparison report to {OOF_DIR}/calibration_report.csv")

source A: selected calibration = 'identity' (id_cv_mae=31.96+/-4.91, chosen_cv_mae=31.96), production params={}
source B: selected calibration = 'identity' (id_cv_mae=133.72+/-12.72, chosen_cv_mae=133.72), production params={}

Saved calibration comparison report to /content/drive/MyDrive/calorie_comp/oof/calibration_report.csv


## 22. Final inference — ensemble of best-fold checkpoints + TTA + calibration + clipping

The submission is produced by the **best models from training** (the
`fold{f}_best.pt` checkpoints selected on validation MAE, not the last epoch),
combined as:

1. **TTA**, applied deterministically to the test cache: `identity` + `hflip`
   for **both** sources; **+90°/180°/270° rotations (each with and without
   hflip)**, i.e. the full 8-element dihedral group, **for Source A only**
   (top-down view is rotation-invariant — same augmentation family used in
   training, §13). Source B only gets the 2-way hflip set, matching its
   training augmentation (small ±10° jitter at train time doesn't warrant
   large-rotation TTA at inference).
2. **Ensemble across all trained fold checkpoints**, averaging every
   (fold × TTA-variant) prediction **in kcal space** (not log space — the
   metric lives in kcal space, and averaging post-`expm1` is what actually
   minimizes expected MAE under the ensemble).
3. **Per-source calibration** from §21 applied to the ensembled raw
   prediction.
4. **Per-source clipping** to `[train_min, train_p99.5]` — computed fresh from
   the training data in this run (not hardcoded), guarding against occasional
   tail blow-ups without arbitrarily capping legitimate high-calorie Source B
   predictions below their observed range.

**Runs on: GPU** (re-runs the forward pass for every fold × TTA-variant × test
image; cheap in absolute terms since test is only 547 images, but still needs
the model on GPU).

In [ ]:
def transform_array(arr: np.ndarray, name: str) -> np.ndarray:
    if name == "identity":
        out = arr
    elif name == "hflip":
        out = arr[:, :, ::-1, :]
    elif name == "rot90":
        out = np.rot90(arr, 1, axes=(1, 2))
    elif name == "rot90_hflip":
        out = np.rot90(arr, 1, axes=(1, 2))[:, :, ::-1, :]
    elif name == "rot180":
        out = np.rot90(arr, 2, axes=(1, 2))
    elif name == "rot180_hflip":
        out = np.rot90(arr, 2, axes=(1, 2))[:, :, ::-1, :]
    elif name == "rot270":
        out = np.rot90(arr, 3, axes=(1, 2))
    elif name == "rot270_hflip":
        out = np.rot90(arr, 3, axes=(1, 2))[:, :, ::-1, :]
    else:
        raise ValueError(name)
    return np.ascontiguousarray(out)

@torch.no_grad()
def predict_kcal_batch(model, cache_array, source_idx_arr, device, batch_size):
    model.eval()
    preds = np.empty(len(cache_array), dtype=np.float32)
    for start in range(0, len(cache_array), batch_size):
        end = min(start + batch_size, len(cache_array))
        imgs = cache_array[start:end]
        x = torch.stack([to_tensor_normalized(img) for img in imgs]).to(device, non_blocking=True)
        src = torch.tensor(source_idx_arr[start:end], dtype=torch.long).to(device, non_blocking=True)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                             dtype=GPU_INFO["amp_dtype"], enabled=device.type == "cuda"):
            z = model(x, src)
        preds[start:end] = torch.expm1(torch.clamp(z, max=LOG1P_CLAMP_MAX)).float().cpu().numpy()
    return preds

ALL_TTA = ["identity", "hflip", "rot90", "rot90_hflip", "rot180", "rot180_hflip", "rot270", "rot270_hflip"]
COMMON_TTA = ["identity", "hflip"]
if CFG.SMOKE_TEST and not IN_COLAB:
    ALL_TTA = ["identity"]
    COMMON_TTA = ["identity"]
elif not CFG.tta_dihedral_A:
    ALL_TTA = [t for t in ALL_TTA if t in COMMON_TTA or (CFG.tta_hflip_B and t == "hflip")]
    if not CFG.tta_hflip_B:
        ALL_TTA = ["identity"]
        COMMON_TTA = ["identity"]
print(f"TTA variants: {ALL_TTA}")

fold_ckpts = sorted(glob.glob(f"{CKPT_DIR}/fold*_best.pt"))
assert len(fold_ckpts) == len(folds_to_run), \
    f"expected {len(folds_to_run)} fold-best checkpoints, found {len(fold_ckpts)}: {fold_ckpts}"
print(f"Ensembling {len(fold_ckpts)} fold checkpoints: {fold_ckpts}")

test_source_idx = test_df["source"].map(SOURCE_TO_IDX).values
is_A = (test_df["source"] == "A").values
accum = np.zeros(len(test_df), dtype=np.float64)
count = np.zeros(len(test_df), dtype=np.float64)

for ckpt_path in fold_ckpts:
    model = build_model().to(device)
    load_checkpoint(ckpt_path, model, restore_rng=False)
    for tta_name in ALL_TTA:
        apply_mask = np.ones(len(test_df), dtype=bool) if tta_name in COMMON_TTA else is_A
        if not apply_mask.any():
            continue
        arr = transform_array(test_cache, tta_name)
        preds = predict_kcal_batch(model, arr, test_source_idx, device, batch_size=GPU_INFO["micro_batch"] * 2)
        accum[apply_mask] += preds[apply_mask]
        count[apply_mask] += 1
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert (count > 0).all()
ensemble_preds_raw = accum / count
print(f"TTA+ensemble done. A gets {count[is_A][0]:.0f} terms/image, B gets {count[~is_A][0]:.0f} terms/image "
      f"({len(fold_ckpts)} folds x TTA-variants-per-source).")

TTA variants: ['identity', 'hflip', 'rot90', 'rot90_hflip', 'rot180', 'rot180_hflip', 'rot270', 'rot270_hflip']
Ensembling 5 fold checkpoints: ['/content/drive/MyDrive/calorie_comp/checkpoints/fold0_best.pt', '/content/drive/MyDrive/calorie_comp/checkpoints/fold1_best.pt', '/content/drive/MyDrive/calorie_comp/checkpoints/fold2_best.pt', '/content/drive/MyDrive/calorie_comp/checkpoints/fold3_best.pt', '/content/drive/MyDrive/calorie_comp/checkpoints/fold4_best.pt']
TTA+ensemble done. A gets 40 terms/image, B gets 10 terms/image (5 folds x TTA-variants-per-source).


## 23. Apply calibration, clip, and write `submission.csv`

Final assembly: per-source calibration (§21) → per-source clip to
`[train_min, train_p99.5]` → assemble in the **exact required format**
(`image_id,predicted_calories`, no extra columns, 547 rows, `test_ids.csv`
order). Under `SMOKE_TEST` the row-count/order assertions are relaxed to match
the smoke subset (a real 547-row submission isn't expected from a toy run),
but every other check (no NaNs, no negatives, exact column names, correct
relative order) still runs.

**Runs on: CPU.**

In [ ]:
calibrated_preds = ensemble_preds_raw.copy()
for src, idx in SOURCE_TO_IDX.items():
    mask = test_source_idx == idx
    calib = selected_calibration[src]
    calibrated_preds[mask] = calib["apply_fn"](ensemble_preds_raw[mask], calib["params"])

clip_bounds = {}
for src in ["A", "B"]:
    sub = train_df.loc[train_df["source"] == src, "calories"]
    clip_bounds[src] = (float(sub.min()), float(sub.quantile(0.995)))
print("Per-source clip bounds [train_min, train_p99.5]:", clip_bounds)

final_preds = calibrated_preds.copy()
for src, idx in SOURCE_TO_IDX.items():
    mask = test_source_idx == idx
    lo, hi = clip_bounds[src]
    final_preds[mask] = np.clip(final_preds[mask], lo, hi)

submission = pd.DataFrame({"image_id": test_df["image_id"].values, "predicted_calories": final_preds})

if not CFG.SMOKE_TEST:
    test_ids_order = pd.read_csv(f"{DATA_DIR}/test_ids.csv")["image_id"].tolist()
    submission = submission.set_index("image_id").loc[test_ids_order].reset_index()
    assert len(submission) == 547, f"expected 547 rows, got {len(submission)}"
    assert list(submission["image_id"]) == test_ids_order, "submission image_id order doesn't match test_ids.csv"
else:
    print("SMOKE_TEST active: skipping the full 547-row / exact test_ids.csv-order assertion "
          "(only a small subset was run end-to-end) -- checking internal consistency instead.")
    assert list(submission["image_id"]) == list(test_df["image_id"])

assert list(submission.columns) == ["image_id", "predicted_calories"]
assert submission["predicted_calories"].isna().sum() == 0, "submission contains NaN predictions"
assert (submission["predicted_calories"] > 0).all(), "submission contains non-positive predictions"

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nWrote submission to {SUBMISSION_PATH} ({len(submission)} rows)")
submission.head()

Per-source clip bounds [train_min, train_p99.5]: {'A': (50.0, 791.3000000000002), 'B': (67.2, 3664.566799999998)}

Wrote submission to /content/drive/MyDrive/calorie_comp/submission.csv (547 rows)


,image_id,predicted_calories
0,test_0000,94.4125
1,test_0001,179.6000
2,test_0002,162.4000
3,test_0003,249.8500
4,test_0004,200.1250


## 24. Done

- `SMOKE_TEST=True`: this run just verified cache → train → OOF → calibration
  → TTA inference → submission writer all work end-to-end on a tiny subset.
  Flip `CFG.SMOKE_TEST = False` in §3 and re-run top to bottom for the real
  pipeline (safe to re-run — cache, weights, and per-fold checkpoints already
  on Drive are reused/resumed rather than rebuilt).
- Full-run deliverable: `PROJECT_DIR/submission.csv`, plus
  `PROJECT_DIR/folds.csv`, `PROJECT_DIR/oof/oof_predictions.csv`,
  `PROJECT_DIR/oof/calibration_report.csv`, and `PROJECT_DIR/logs/cv_results.csv`
  as supporting artifacts documenting the CV methodology and results.